In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 11
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## Fraser EFW Pipeline

**Source:** Fraser Institute Economic Freedom of the World
**Access:** Automated — scrapes annual report page to find XLSX download URL
**Download instructions:** See `docs/instructions_data_maintenance.md` — FRASER section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Area 2: Legal System and Property Rights | Property rights / Judicial independence | Primary tier 1 |
| Area 4: Freedom to Trade Internationally | Trade openness | Primary tier 2 |
| Area 5: Regulation | Regulatory quality | Primary tier 1 |

In [2]:
import re
import io
import requests
import pandas as pd
from datetime import datetime

HEADERS = BROWSER_HEADERS

def get_fraser_url():
    """
    Scrape the latest Fraser EFW annual report page to find the XLSX download URL.
    Tries current year and falls back to prior years.
    """
    current_year = datetime.today().year
    for year in range(current_year, current_year - 3, -1):
        page_url = f"https://www.fraserinstitute.org/studies/economic-freedom-world-{year}-annual-report"
        response = requests.get(page_url, headers=HEADERS, timeout=30)
        if response.status_code != 200:
            continue
        xlsx_links = re.findall(r'https?://[^\s"\'<>]+\.xlsx', response.text)
        # Filter to master index data file
        data_links = [l for l in xlsx_links if 'master-index' in l or 'dataset' in l or 'researchers' in l]
        if data_links:
            print(f"Found Fraser EFW {year} data file: {data_links[0]}")
            return data_links[0], year
    return None, None

FRASER_URL, FRASER_YEAR = get_fraser_url()

if FRASER_URL:
    print(f"\nDownloading Fraser EFW {FRASER_YEAR}...")
    response = requests.get(FRASER_URL, headers=HEADERS, timeout=60)
    print(f"Status: {response.status_code}, Size: {len(response.content)/1024:.1f}KB")
    xl = pd.ExcelFile(io.BytesIO(response.content), engine='openpyxl')
    print(f"Sheets: {xl.sheet_names}")
else:
    print("Could not find Fraser EFW download URL")

Found Fraser EFW 2025 data file: https://www.fraserinstitute.org/sites/default/files/2025-09/economic-freedom-of-the-world-2025-master-index-data-for-researchers-iso.xlsx

Status: 200, Size: 5292.4KB
Sheets: ['EFW Index 1970-2023', 'EFW Panel Dataset', 'EFW Ratings 1950-1965']


In [3]:
# Inspect the panel dataset sheet
df_panel = xl.parse('EFW Panel Dataset')
print(f"Shape: {df_panel.shape}")
print(f"Columns: {list(df_panel.columns[:15])}")
print(df_panel.head(3))

Shape: (4950, 12)
Columns: ['ISO_Code', 'Countries', 'Year', 'Summary', 'Area 1', 'Area 2', 'Area 3', 'Area 4', 'Area 5', 'Standard Deviation of the 5 EFW Areas', 'World Bank Region', 'World Bank Current Income Classification, 1990-Present']
  ISO_Code Countries  Year  Summary    Area 1    Area 2    Area 3    Area 4  \
0      ALB   Albania  2023     7.56  7.637373  5.552222  8.948484  8.536164   
1      DZA   Algeria  2023     4.24  4.296913  3.788305  6.382111  2.593994   
2      AGO    Angola  2023     5.16  7.834422  3.477577  4.378911  4.971711   

     Area 5  Standard Deviation of the 5 EFW Areas  \
0  7.108518                               1.334328   
1  4.118263                               1.371129   
2  5.135053                               1.630127   

            World Bank Region  \
0       Europe & Central Asia   
1  Middle East & North Africa   
2          Sub-Saharan Africa   

  World Bank Current Income Classification, 1990-Present  
0                               

In [4]:
print(f"Years: {sorted(df_panel['Year'].unique())}")
print(f"Countries: {df_panel['Countries'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (df_panel.isnull().sum() / len(df_panel) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

Years: [np.int64(1970), np.int64(1975), np.int64(1980), np.int64(1985), np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Countries: 165

Missing values (%):
World Bank Current Income Classification, 1990-Present    16.7
Area 4                                                    16.4
Summary                                                    7.9
Standard Deviation of the 5 EFW Areas                      7.9
World Bank Region                                          7.9
Area 3                                                     6.6
Area 1                                                     5.4
Area 5                             

In [5]:
# Filter to framework indicators and years
fraser = df_panel[['ISO_Code', 'Countries', 'Year', 'Area 2', 'Area 4', 'Area 5']].copy()

# Rename columns
fraser = fraser.rename(columns={
    'ISO_Code':  'country_code',
    'Countries': 'country_name',
    'Year':      'year',
    'Area 2':    'fraser_legal_system',
    'Area 4':    'fraser_trade_freedom',
    'Area 5':    'fraser_regulation',
})

# Filter to framework start year
fraser = fraser[fraser['year'] >= FRAMEWORK_START_YEAR].copy()
fraser = fraser.sort_values(['country_code', 'year']).reset_index(drop=True)

print(f"Shape: {fraser.shape}")
print(f"Years: {sorted(fraser['year'].unique())}")
print(f"Countries: {fraser['country_code'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (fraser.isnull().sum() / len(fraser) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(fraser.head())

Shape: (4290, 6)
Years: [np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Countries: 165

Missing values (%):
fraser_trade_freedom    11.6
fraser_legal_system      1.5
fraser_regulation        1.0
dtype: float64
  country_code country_name  year  fraser_legal_system  fraser_trade_freedom  \
0          AGO       Angola  1990             4.181102                   NaN   
1          AGO       Angola  1995             3.743334                   NaN   
2          AGO       Angola  2000             3.151128                   NaN   
3          AGO       Angola  2001             3.153537                   NaN   
4          AGO       

In [6]:
# Get latest year from data
latest_year = str(int(fraser['year'].max()))
data_as_of_date = latest_year

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "fraser_clean.csv")
fraser.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {fraser.shape}")

# Update download log
update_entry(
    "FRASER_REG",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="fraser_clean.csv",
    latest_available_version=f"EFW {FRASER_YEAR}",
    notes=f"Areas 2 (Legal System), 4 (Trade Freedom), 5 (Regulation). Auto-detects latest annual report page. Panel dataset chain-linked. Coverage: 1990-{latest_year}, 165 countries. Annual from 2000, quinquennial before."
)

update_entry(
    "FRASER_LEGAL",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="fraser_clean.csv",
    latest_available_version=f"EFW {FRASER_YEAR}",
    notes="Same file as FRASER_REG — fraser_clean.csv. Area 2: Legal System and Property Rights."
)

print_entry("FRASER_REG")
print_entry("FRASER_LEGAL")

Written: /Users/boulanger/Documents/governance-framework/data/processed/fraser_clean.csv
Shape: (4290, 6)
[download_log] Updated entry for FRASER_REG
[download_log] Updated entry for FRASER_LEGAL
  source_id: FRASER_REG
  last_attempted_date: 2026-06-02
  last_successful_download_date: 2026-06-02
  data_as_of_date: 2023
  local_filename: fraser_clean.csv
  latest_available_version: EFW 2025
  no_update_reason: nan
  notes: Areas 2 (Legal System), 4 (Trade Freedom), 5 (Regulation). Auto-detects latest annual report page. Panel dataset chain-linked. Coverage: 1990-2023, 165 countries. Annual from 2000, quinquennial before.
  source_id: FRASER_LEGAL
  last_attempted_date: 2026-06-02
  last_successful_download_date: 2026-06-02
  data_as_of_date: 2023
  local_filename: fraser_clean.csv
  latest_available_version: EFW 2025
  no_update_reason: nan
  notes: Same file as FRASER_REG — fraser_clean.csv. Area 2: Legal System and Property Rights.


In [7]:
import re

HEADERS = BROWSER_HEADERS

# Test Heritage download page for direct Excel links
response = requests.get(
    "https://indexdotnet.azurewebsites.net/index/download",
    headers=HEADERS,
    timeout=30
)
print(f"Status: {response.status_code}")
xlsx_links = re.findall(r'https?://[^\s"\'<>]+\.xlsx', response.text)
csv_links = re.findall(r'https?://[^\s"\'<>]+\.csv', response.text)
print(f"XLSX links: {xlsx_links}")
print(f"CSV links: {csv_links}")

Status: 200
XLSX links: []
CSV links: []


In [9]:
# Try economicfreedom.heritage.org for data links
test_urls = [
    "https://economicfreedom.heritage.org/api/countries/scores",
    "https://economicfreedom.heritage.org/api/index/download",
    "https://static.heritage.org/index/excel/2026/heritage_index_2026.xlsx",
    "https://static.heritage.org/index/excel/2025/heritage_index_2025.xlsx",
]

for url in test_urls:
    try:
        response = requests.head(url, headers=HEADERS, timeout=10, allow_redirects=True)
        content_type = response.headers.get('Content-Type', '')
        print(f"{response.status_code}: {url} [{content_type[:40]}]")
    except Exception as e:
        print(f"ERROR: {url} — {e}")

403: https://economicfreedom.heritage.org/api/countries/scores [text/html; charset=UTF-8]
403: https://economicfreedom.heritage.org/api/index/download [text/html; charset=UTF-8]
404: https://static.heritage.org/index/excel/2026/heritage_index_2026.xlsx [text/plain;charset=UTF-8]
404: https://static.heritage.org/index/excel/2025/heritage_index_2025.xlsx [text/plain;charset=UTF-8]
